1. Importar librerías necesarias

Importamos las librerías esenciales para conectar a las bases de datos PostgreSQL, manipular datos con pandas y leer configuración desde YAML.
Python

In [23]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

2. Cargar configuración y crear conexiones

Leemos el archivo config.yml para obtener las credenciales de ambas bases de datos (origen MENSAJERIA_OLTP y destino ETL_PROCESS). Creamos dos motores SQLAlchemy para conectarnos a cada una.

In [24]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

3. Extraer y fusionar datos de clientes

Leemos las tablas cliente, tipo_cliente y ciudad de la base de datos OLTP. Realizamos dos fusiones (merges) para traer el nombre del tipo de cliente y el nombre de la ciudad principal, aplanando así nuestra dimensión para la bodega de datos.

In [25]:
# Extraemos las tablas necesarias
cliente = pd.read_sql_table('cliente', mensajeria)
tipo_cliente = pd.read_sql_table('tipo_cliente', mensajeria)
ciudad = pd.read_sql_table('ciudad', mensajeria)

# Seleccionamos solo las columnas que nos interesan de las tablas complementarias
tipo_cliente = tipo_cliente[['tipo_cliente_id', 'nombre']]
ciudad = ciudad[['ciudad_id', 'nombre']]

# Primer merge: unimos cliente con tipo_cliente
dim_cliente = cliente.merge(
    tipo_cliente,
    on='tipo_cliente_id',
    how='left'
)

# Renombramos temporalmente las columnas para no confundir los 'nombre'
dim_cliente = dim_cliente.rename(columns={'nombre_x': 'nombre_cliente', 'nombre_y': 'tipo_cliente'})

# Segundo merge: unimos con ciudad para traer el nombre de la ciudad principal
dim_cliente = dim_cliente.merge(
    ciudad,
    on='ciudad_id',
    how='left'
)

dim_cliente = dim_cliente.rename(columns={'nombre': 'ciudad_principal', 'activo': 'estado_activo'})

dim_cliente

,cliente_id,nit_cliente,nombre_cliente,email,direccion,telefono,nombre_contacto,ciudad_id,tipo_cliente_id,estado_activo,coordinador_id,sector,tipo_cliente,ciudad_principal
0,1,25,Cliente 2,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,S,Persona Juridica,CALI
1,2,123,Cliente 1,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,2.0,industrial,Persona Juridica,CALI
2,6,24390-3,CLINICA DEPORTIVA DEL SUR,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,1.0,salud,Persona Juridica,CALI
3,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
4,8,5017350-8,CLINICA NEFROLOGOS DE CALI,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
5,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
6,4,306215-0,CRUZ AZUL-LIFE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
7,5,300513-3,CLINICA CALI -JOVEN,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
8,7,951033-8,CLINICA COFFE -HEALTH,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI
9,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,algo.com,Calle 100 No 25-18,327-00000,Cristiano Ronaldo,1,1,True,NaN,salud,Persona Juridica,CALI


4. Limpiar y transformar datos

Seleccionamos únicamente las columnas que nos interesan para nuestra bodega de datos, eliminamos duplicados, ordenamos por ID, reseteamos los índices y creamos una nueva columna cliente_key con valores secuenciales (Surrogate Key). Finalmente, rellenamos valores nulos en el sector o ciudad si los hubiera.

In [26]:
# Seleccionamos las columnas definitivas para la dimensión
columnas_deseadas = [
    'cliente_id', 'nit_cliente', 'nombre_cliente', 
    'sector', 'tipo_cliente', 'ciudad_principal', 'estado_activo'
]
dim_cliente = dim_cliente[columnas_deseadas]

dim_cliente = (
    dim_cliente
    .drop_duplicates(subset=['cliente_id'])
    .sort_values('cliente_id')
    .reset_index(drop=True)
)

# Creamos la llave sustituta (Surrogate Key) para la bodega de datos
dim_cliente['cliente_key'] = dim_cliente.index + 1

# Tratamiento de nulos (opcional, dependiendo de la calidad de tus datos)
dim_cliente['sector'] = dim_cliente['sector'].fillna('No especificado')
dim_cliente['ciudad_principal'] = dim_cliente['ciudad_principal'].fillna('Sin ciudad')

# Reordenamos dejando la llave sustituta de primera
dim_cliente = dim_cliente[['cliente_key', 'cliente_id', 'nit_cliente', 'nombre_cliente', 'sector', 'tipo_cliente', 'ciudad_principal', 'estado_activo']]

dim_cliente

,cliente_key,cliente_id,nit_cliente,nombre_cliente,sector,tipo_cliente,ciudad_principal,estado_activo
0,1,1,25,Cliente 2,S,Persona Juridica,CALI,True
1,2,2,123,Cliente 1,industrial,Persona Juridica,CALI,True
2,3,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,salud,Persona Juridica,CALI,True
3,4,4,306215-0,CRUZ AZUL-LIFE,salud,Persona Juridica,CALI,True
4,5,5,300513-3,CLINICA CALI -JOVEN,salud,Persona Juridica,CALI,True
5,6,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud,Persona Juridica,CALI,True
6,7,7,951033-8,CLINICA COFFE -HEALTH,salud,Persona Juridica,CALI,True
7,8,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,Persona Juridica,CALI,True
8,9,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,salud,Persona Juridica,CALI,True
9,10,10,550254-8,CLINICA VIDA Y SALUD,salud,Persona Juridica,CALI,True


5. Limpiar tabla destino e insertar datos

Primero, vaciamos la tabla dim_cliente en la base de datos ETL (truncate). Luego, insertamos el dataframe limpio y transformado en esa tabla. Esto asegura que los datos estén actualizados sin duplicados.

In [28]:
with etl_conn.connect() as conn:
    conn.execute(db.text("TRUNCATE TABLE dim_cliente RESTART IDENTITY CASCADE"))
    conn.commit()

# Insertamos los datos transformados
dim_cliente.to_sql('dim_cliente', etl_conn, if_exists='append', index=False)

dim_cliente

,cliente_key,cliente_id,nit_cliente,nombre_cliente,sector,tipo_cliente,ciudad_principal,estado_activo
0,1,1,25,Cliente 2,S,Persona Juridica,CALI,True
1,2,2,123,Cliente 1,industrial,Persona Juridica,CALI,True
2,3,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,salud,Persona Juridica,CALI,True
3,4,4,306215-0,CRUZ AZUL-LIFE,salud,Persona Juridica,CALI,True
4,5,5,300513-3,CLINICA CALI -JOVEN,salud,Persona Juridica,CALI,True
5,6,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud,Persona Juridica,CALI,True
6,7,7,951033-8,CLINICA COFFE -HEALTH,salud,Persona Juridica,CALI,True
7,8,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,Persona Juridica,CALI,True
8,9,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,salud,Persona Juridica,CALI,True
9,10,10,550254-8,CLINICA VIDA Y SALUD,salud,Persona Juridica,CALI,True


6. Verificar datos insertados

Ejecutamos una consulta SELECT para confirmar que los datos fueron insertados correctamente en la base de datos ETL.

In [29]:
# Verificamos la tabla en la bodega de datos
pd.read_sql('SELECT * FROM dim_cliente LIMIT 10', etl_conn)

,cliente_key,cliente_id,nit_cliente,nombre_cliente,sector,tipo_cliente,ciudad_principal,estado_activo
0,1,1,25,Cliente 2,S,Persona Juridica,CALI,True
1,2,2,123,Cliente 1,industrial,Persona Juridica,CALI,True
2,3,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,salud,Persona Juridica,CALI,True
3,4,4,306215-0,CRUZ AZUL-LIFE,salud,Persona Juridica,CALI,True
4,5,5,300513-3,CLINICA CALI -JOVEN,salud,Persona Juridica,CALI,True
5,6,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud,Persona Juridica,CALI,True
6,7,7,951033-8,CLINICA COFFE -HEALTH,salud,Persona Juridica,CALI,True
7,8,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud,Persona Juridica,CALI,True
8,9,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,salud,Persona Juridica,CALI,True
9,10,10,550254-8,CLINICA VIDA Y SALUD,salud,Persona Juridica,CALI,True
